# 🔬 Notebook 3: Key-Value Store — Deep Dives


## 🛠️ Setup

```bash
cd 06-system-designs/key-value-store
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Deep dive 1 — consistent hashing (with and without vnodes)

### 😱 Bad: one position per node

When each node has a single ring position, load imbalance gets ugly with small N:
a node that happens to sit just after a long empty arc owns *all* those keys.


In [1]:
import hashlib, bisect
from collections import defaultdict

def h(s: str) -> int:
    return int(hashlib.md5(s.encode()).hexdigest(), 16)

class Ring:
    def __init__(self, vnodes_per_node: int = 1):
        self.vnodes_per_node = vnodes_per_node
        self.ring: list[tuple[int, str]] = []
        self.positions: list[int] = []

    def add_node(self, node: str):
        for i in range(self.vnodes_per_node):
            self.ring.append((h(f"{node}#{i}"), node))
        self.ring.sort()
        self.positions = [p for p, _ in self.ring]

    def owner(self, key: str) -> str:
        i = bisect.bisect_right(self.positions, h(key)) % len(self.ring)
        return self.ring[i][1]

    def replicas(self, key: str, n: int) -> list[str]:
        i = bisect.bisect_right(self.positions, h(key)) % len(self.ring)
        out, seen = [], set()
        while len(out) < n and len(seen) < len(set(n for _, n in self.ring)):
            node = self.ring[i % len(self.ring)][1]
            if node not in seen:
                seen.add(node); out.append(node)
            i += 1
        return out

def load_dist(ring, n_keys=20_000):
    d = defaultdict(int)
    for k in range(n_keys):
        d[ring.owner(f"k{k}")] += 1
    return d

bad = Ring(vnodes_per_node=1)
for node in ["A", "B", "C", "D", "E"]:
    bad.add_node(node)
dist = load_dist(bad)
print("1 vnode / node  →", dict(dist))
print(f"   spread: min={min(dist.values())}  max={max(dist.values())}  "
      f"ratio={max(dist.values())/min(dist.values()):.2f}x")

1 vnode / node  → {'D': 7711, 'C': 7927, 'A': 609, 'E': 3483, 'B': 270}
   spread: min=270  max=7927  ratio=29.36x


### ✅ Best: many vnodes per node smooths the load

In [2]:
good = Ring(vnodes_per_node=128)
for node in ["A", "B", "C", "D", "E"]:
    good.add_node(node)
dist = load_dist(good)
print("128 vnodes / node →", dict(dist))
print(f"   spread: min={min(dist.values())}  max={max(dist.values())}  "
      f"ratio={max(dist.values())/min(dist.values()):.2f}x")
print("➡ Much more even. Also: removing a node redistributes its share "
      "across ALL remaining nodes, not just one unlucky neighbor.")

128 vnodes / node → {'A': 4157, 'E': 3781, 'B': 4567, 'C': 4226, 'D': 3269}
   spread: min=3269  max=4567  ratio=1.40x
➡ Much more even. Also: removing a node redistributes its share across ALL remaining nodes, not just one unlucky neighbor.


## Deep dive 2 — a tiny quorum simulator

We simulate N replicas that sometimes drop requests (`fail_prob`). We try different
W/R settings and watch what happens.


In [3]:
import random

class Replica:
    def __init__(self, name, fail_prob=0.0):
        self.name = name
        self.store: dict[str, tuple[str, int]] = {}
        self.fail_prob = fail_prob

    def write(self, key, value, version):
        if random.random() < self.fail_prob:
            return False
        cur = self.store.get(key)
        if cur is None or version > cur[1]:
            self.store[key] = (value, version)
        return True

    def read(self, key):
        if random.random() < self.fail_prob:
            return None
        return self.store.get(key)

class Coordinator:
    def __init__(self, replicas, N, W, R):
        self.replicas = replicas
        self.N, self.W, self.R = N, W, R
        self.version = 0

    def put(self, key, value):
        self.version += 1
        acks = 0
        for r in self.replicas:
            if r.write(key, value, self.version):
                acks += 1
                if acks >= self.W:
                    return {"status": "ok", "acks": acks}
        return {"status": "fail", "acks": acks}

    def get(self, key):
        responses = []
        for r in self.replicas:
            v = r.read(key)
            if v is not None:
                responses.append(v)
                if len(responses) >= self.R:
                    break
        if not responses:
            return None
        return max(responses, key=lambda x: x[1])   # pick highest version

random.seed(0)
replicas = [Replica(f"r{i}", fail_prob=0.2) for i in range(3)]

# Configuration A: quorum (W=2, R=2)
cA = Coordinator(replicas, N=3, W=2, R=2)
fails_A = sum(1 for _ in range(200) if cA.put(f"kA{_}", "v")["status"] == "fail")
print(f"W=2 R=2 (quorum):   {fails_A}/200 writes failed")

# Configuration B: W=3 — every replica must ack, so any drop = failure
cB = Coordinator(replicas, N=3, W=3, R=1)
fails_B = sum(1 for _ in range(200) if cB.put(f"kB{_}", "v")["status"] == "fail")
print(f"W=3 R=1 (strict):   {fails_B}/200 writes failed")
print("➡ Higher W = stronger guarantees, lower availability. That is CAP in one line.")

W=2 R=2 (quorum):   23/200 writes failed
W=3 R=1 (strict):   91/200 writes failed
➡ Higher W = stronger guarantees, lower availability. That is CAP in one line.


## Deep dive 3 — anti-entropy with Merkle trees

Two replicas **drift** apart (a node was down; writes went to its peers). When it
returns, we want to find the **missing keys** without comparing the entire dataset.

A Merkle tree builds a hash tree over the keyspace. Two replicas compare their
roots; if roots match, they're in sync — **zero** key comparisons. If roots
differ, we recurse **only into differing subtrees**. For small diffs, bandwidth
is `O(log N)` instead of `O(N)`.

```
        H(root)
        /     \
     H(L)     H(R)      ← compare roots; differ → recurse
     /  \     /  \
   ...  ...  ...  ...    ← only descend into differing subtrees
```

Cassandra uses this for **repair**, DynamoDB and Riak for **hinted handoff / read repair**.
Let's build one:


In [4]:
import hashlib
from dataclasses import dataclass, field

def H(b: bytes) -> str:
    return hashlib.sha1(b).hexdigest()[:10]   # short hashes for readable output

@dataclass
class MerkleNode:
    hash: str
    left: "MerkleNode | None" = None
    right: "MerkleNode | None" = None
    keys: list[str] = field(default_factory=list)   # only on leaves

def build_merkle(items: dict[str, str], bucket_size: int = 4) -> MerkleNode:
    """Bucket keys by sorted order, hash each bucket as a leaf, then fold up."""
    sorted_items = sorted(items.items())
    # Leaves
    leaves = []
    for i in range(0, len(sorted_items), bucket_size):
        chunk = sorted_items[i:i+bucket_size]
        payload = "|".join(f"{k}={v}" for k, v in chunk).encode()
        leaves.append(MerkleNode(H(payload), keys=[k for k, _ in chunk]))
    if not leaves:
        return MerkleNode(H(b""))
    # Fold pairwise until one root
    while len(leaves) > 1:
        nxt = []
        for i in range(0, len(leaves), 2):
            l = leaves[i]
            r = leaves[i+1] if i+1 < len(leaves) else leaves[i]
            nxt.append(MerkleNode(H((l.hash + r.hash).encode()), left=l, right=r))
        leaves = nxt
    return leaves[0]

def diff(a: MerkleNode, b: MerkleNode, out: list[str]) -> int:
    """Return number of leaf comparisons performed; collect differing keys."""
    if a.hash == b.hash:
        return 1                                  # 1 comparison, done
    if a.left is None and b.left is None:         # both leaves
        out.extend(set(a.keys) | set(b.keys))
        return 1
    return 1 + diff(a.left, b.left, out) + diff(a.right, b.right, out)

# Replica A and B differ on just 2 keys out of 32
A = {f"k{i:02d}": f"v{i}"          for i in range(32)}
B = dict(A)
B["k05"] = "v5-CHANGED"
B["k20"] = "v20-CHANGED"

tA, tB = build_merkle(A), build_merkle(B)
differing, comparisons = [], 0
comparisons = diff(tA, tB, differing)
print(f"Naive comparison:  would need 32 key checks.")
print(f"Merkle tree:       {comparisons} node-hash comparisons.")
print(f"Detected drift on: {sorted(set(differing))}")

Naive comparison:  would need 32 key checks.
Merkle tree:       11 node-hash comparisons.
Detected drift on: ['k04', 'k05', 'k06', 'k07', 'k20', 'k21', 'k22', 'k23']


## Deep dive 4 — hinted handoff + read repair

What if a replica is temporarily down when we try to write? We don't fail — we
write to a **substitute** node with a "hint" saying *"this really belongs to N3;
deliver it when N3 comes back"*. When N3 returns, the hint is replayed.

**Read repair** is the other side: on every read, if replicas return different
versions, the coordinator pushes the newest version back to the stale ones.

Tiny simulation:


In [5]:
class NodeWithHints:
    def __init__(self, name):
        self.name = name
        self.store: dict[str, tuple[str, int]] = {}
        self.hints: list[tuple[str, str, str, int]] = []   # (target, k, v, ver)
        self.alive = True

    def write(self, k, v, ver):
        if not self.alive:
            return False
        cur = self.store.get(k)
        if cur is None or ver > cur[1]:
            self.store[k] = (v, ver)
        return True

    def store_hint(self, target, k, v, ver):
        self.hints.append((target, k, v, ver))

    def flush_hints(self, cluster):
        delivered = 0
        remaining = []
        for target, k, v, ver in self.hints:
            if cluster[target].write(k, v, ver):
                delivered += 1
            else:
                remaining.append((target, k, v, ver))
        self.hints = remaining
        return delivered

nodes = {n: NodeWithHints(n) for n in ["A", "B", "C"]}
nodes["C"].alive = False                   # C is down

# Client writes key "x" — replicas are [A, B, C]. C is down → A takes a hint for C.
ver = 1
nodes["A"].write("x", "hello", ver)
nodes["B"].write("x", "hello", ver)
nodes["A"].store_hint("C", "x", "hello", ver)
print("C store before recovery:", nodes["C"].store)
print("A holds hints:          ", nodes["A"].hints)

# C comes back — A replays hints to it
nodes["C"].alive = True
delivered = nodes["A"].flush_hints(nodes)
print(f"After recovery: {delivered} hint(s) delivered")
print("C store after recovery: ", nodes["C"].store)

C store before recovery: {}
A holds hints:           [('C', 'x', 'hello', 1)]
After recovery: 1 hint(s) delivered
C store after recovery:  {'x': ('hello', 1)}


## Deep dive 5 — putting it all together

A production read path in this design looks like:

1. Client hits any node → that node is the **coordinator**.
2. Coordinator hashes the key, finds the **N replicas** on the ring.
3. It sends the read to all N, waits for **R** responses.
4. If responses disagree, it **reconciles** (pick highest version / return siblings).
5. It **read-repairs** stale replicas in the background.
6. Anti-entropy (Merkle trees) runs periodically to catch anything reads missed.

Write path is symmetric: hash → N replicas → wait for W acks → store hints for
any replica that didn't answer.

That's the whole Dynamo paper in one page. 🎉

### Where to go next
- [Amazon's Dynamo paper (2007)](https://www.allthingsdistributed.com/files/amazon-dynamo-sosp2007.pdf)
- Cassandra docs on [consistent hashing](https://cassandra.apache.org/doc/stable/cassandra/architecture/dynamo.html)
- Designing Data-Intensive Applications, Chapter 5–6 (replication & partitioning)
